# Forma Active — Shopify ETL Pipeline

**Pipeline:** Shopify API → ETL / Synthetic Scaling → PostgreSQL Load → Cohort Analysis  
**Dataset:** 10 real Shopify orders → scaled to 300 simulated orders maintaining FK integrity  
**Tables produced:** `orders`, `customers`, `order_line_items`, `products`

## Portfolio Note

This notebook is a cleaned and consolidated version of the original development workflow.

The original project was developed across multiple exploratory notebooks during a 6–7 day iterative build process involving API extraction, ETL debugging, schema validation, and SQL analytics development.

For portfolio readability, repetitive debugging steps, failed experiments, and intermediate validation cells were removed while preserving the final analytical pipeline and methodology.

## 1. Imports & Shopify API Connection

In [ ]:
import requests
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

# Shopify credentials (redacted for portfolio)
SHOP  = "forma-active-dev.myshopify.com"
TOKEN = "YOUR_ACCESS_TOKEN"

In [ ]:
def fetch_all_orders(shop, token):
    """Paginate through Shopify Admin API and return all orders."""
    url = f"https://{shop}/admin/api/2024-01/orders.json?limit=250&status=any"
    headers = {"X-Shopify-Access-Token": token}
    all_orders = []

    while url:
        response = requests.get(url, headers=headers)
        data = response.json()
        all_orders.extend(data.get("orders", []))
        link = response.headers.get("Link", "")
        if 'rel="next"' in link:
            url = [l.split(";")[0].strip("<> ") for l in link.split(",") if 'rel="next"' in l][0]
        else:
            url = None

    return all_orders

# Original extraction performed via Shopify Admin API
# Raw JSON export preserved locally for reproducibility

import json

# with open("data/raw/shopify_orders_raw.json", "r") as f:
#    orders_raw = json.load(f)
# NOTE: Original Shopify API extraction omitted for security/reproducibility reasons.
# Final cleaned datasets used throughout the project are preserved in /data/.

# Example:
# orders_raw = fetch_all_orders(SHOP, TOKEN)

print(f"Loaded {len(orders_raw)} raw Shopify orders")


## 2. ETL — Flatten Nested JSON into Relational Tables

In [ ]:
# Flatten orders JSON
df_orders = pd.json_normalize(orders_raw)
print("Raw orders shape:", df_orders.shape)

# Extract compact order table — keep only analytical columns
df_orders["order_id"]    = df_orders["id"]
df_orders["customer_id"] = df_orders["customer.id"]
df_orders["created_at"]  = pd.to_datetime(df_orders["created_at"])
df_orders["total_price"] = pd.to_numeric(df_orders["total_price"])

orders_base = df_orders[["order_id", "customer_id", "created_at", "total_price", "financial_status", "line_items"]].copy()
orders_base.head(3)

In [ ]:
# Explode line_items (one row per item) then json_normalize the dict
df_exploded = orders_base.explode("line_items").reset_index(drop=True)
line_items_norm = pd.json_normalize(df_exploded["line_items"])

df_li = pd.concat(
    [df_exploded.drop(columns=["line_items"]).reset_index(drop=True), line_items_norm],
    axis=1
)

# Build clean order_line_items table
order_line_items = df_li[[
    "order_id", "created_at", "product_id", "variant_id",
    "title", "quantity", "price", "total_price"
]].copy()

order_line_items["revenue"] = (
    order_line_items["quantity"].astype(float)
    * order_line_items["price"].astype(float)
)

# Type cleanup
order_line_items["order_id"]    = order_line_items["order_id"].astype(str)
order_line_items["product_id"]  = order_line_items["product_id"].astype("Int64").astype(str)
order_line_items["variant_id"]  = order_line_items["variant_id"].astype("Int64").astype(str)
order_line_items["quantity"]    = order_line_items["quantity"].astype(int)
order_line_items["price"]       = order_line_items["price"].astype(float)
order_line_items                = order_line_items.dropna(subset=["product_id"])

print(f"Real line items: {order_line_items.shape[0]} rows, {order_line_items['order_id'].nunique()} orders")
print(f"Revenue total: £{order_line_items['revenue'].sum():,.2f}")
order_line_items.head(3)

## 3. Synthetic Scaling — 10 → 300 Orders

The dev store contains only 10 real orders. To build a meaningful analytics dataset,  
orders are duplicated ×30 with new synthetic IDs and date offsets, preserving FK integrity.

In [ ]:
np.random.seed(42)
MULTIPLIER = 30

# ── Orders ──────────────────────────────────────────────────────────────────
orders_expanded = pd.concat([orders_base.drop(columns=["line_items"])] * MULTIPLIER, ignore_index=True)
orders_expanded["order_id"]   = "SIM-" + orders_expanded.index.astype(str)
orders_expanded["created_at"] = pd.to_datetime(orders_expanded["created_at"]) + pd.to_timedelta(
    np.random.randint(-72, 72, len(orders_expanded)), unit="h"
)
orders_expanded["total_price"] = pd.to_numeric(orders_expanded["total_price"], errors="coerce")

print(f"Expanded orders: {orders_expanded.shape[0]} rows, {orders_expanded['order_id'].nunique()} unique")

In [ ]:
# ── Line items — rebuild from raw JSON at expanded scale ────────────────────
expanded_line_items = []

for _, order in orders_expanded.iterrows():
    # look up original line_items by matching real order_id suffix
    real_idx = int(order["order_id"].replace("SIM-", "")) % len(orders_raw)
    for item in orders_raw[real_idx].get("line_items", []):
        expanded_line_items.append({
            "order_id":   order["order_id"],
            "created_at": order["created_at"],
            "product_id": item.get("product_id"),
            "variant_id": item.get("variant_id"),
            "title":      item.get("title"),
            "quantity":   item.get("quantity"),
            "price":      float(item.get("price", 0)),
        })

df_line_items_expanded = pd.DataFrame(expanded_line_items)
df_line_items_expanded["revenue"] = df_line_items_expanded["quantity"] * df_line_items_expanded["price"]

print(f"Expanded line items: {df_line_items_expanded.shape[0]} rows")
print(f"Revenue total: £{df_line_items_expanded['revenue'].sum():,.2f}")

In [ ]:
# ── Final clean tables ───────────────────────────────────────────────────────

# orders_clean
orders_clean = orders_expanded[[
    "order_id", "created_at", "customer_id", "financial_status", "total_price"
]].copy()
orders_clean["created_at"]  = pd.to_datetime(orders_clean["created_at"])
orders_clean["order_month"] = orders_clean["created_at"].dt.to_period("M").astype(str)

# Assign 12 synthetic customers (original store only has 3)
customer_pool = [f"CUST-{i:03d}" for i in range(12)]
orders_clean["customer_id"] = np.random.choice(customer_pool, size=len(orders_clean), replace=True)

# order_line_items
order_line_items = df_line_items_expanded[[
    "order_id", "created_at", "product_id", "variant_id",
    "title", "quantity", "price", "revenue"
]].copy()
order_line_items["order_month"]  = pd.to_datetime(order_line_items["created_at"]).dt.to_period("M").astype(str)

# products_variants dimension
products_variants = order_line_items[["product_id", "variant_id", "title", "price"]].drop_duplicates()

# customers_clean
customers_clean = orders_clean.groupby("customer_id").agg(
    first_order   = ("created_at", "min"),
    total_orders  = ("order_id",   "nunique"),
    lifetime_value= ("total_price","sum")
).reset_index()
customers_clean["cohort_month"] = pd.to_datetime(customers_clean["first_order"]).dt.to_period("M").astype(str)

print("=== Final Table Shapes ===")
print(f"orders_clean:      {orders_clean.shape}")
print(f"order_line_items:  {order_line_items.shape}")
print(f"customers_clean:   {customers_clean.shape}")
print(f"products_variants: {products_variants.shape}")

In [ ]:
# ── Referential integrity checks ────────────────────────────────────────────
assert set(orders_clean["customer_id"]).issubset(set(customers_clean["customer_id"])), \
    "orders → customers: FK violation"
assert orders_clean["order_id"].nunique() == len(orders_clean), \
    "orders: duplicate order_ids"

print("All integrity checks passed ✓")
print(f"Unique orders:    {orders_clean['order_id'].nunique()}")
print(f"Unique customers: {customers_clean['customer_id'].nunique()}")
print(f"Unique products:  {products_variants['product_id'].nunique()}")

## 4. PostgreSQL Load — Neon

In [ ]:
# Connection string redacted for security
CONN_STRING = "postgresql://<user>:<password>@<host>/<database>"
engine = create_engine(CONN_STRING, pool_pre_ping=True)

# Verify connection
with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).fetchone()[0][:50])

In [ ]:
# Convert any remaining Period columns to str before loading
def period_to_str(df):
    for col in df.columns:
        if str(df[col].dtype).startswith("period"):
            df[col] = df[col].astype(str)
    return df

tables = {
    "customers":        period_to_str(customers_clean.copy()),
    "orders":           period_to_str(orders_clean.copy()),
    "order_line_items": period_to_str(order_line_items.copy()),
    "products":         period_to_str(products_variants.copy()),
}

for name, df in tables.items():
    df.to_sql(name, engine, if_exists="replace", index=False)
    print(f"Loaded '{name}': {len(df):,} rows")

In [ ]:
# Verify row counts in Postgres
with engine.connect() as conn:
    for tbl in ["customers", "orders", "order_line_items", "products"]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).fetchone()[0]
        print(f"{tbl}: {count:,} rows")

## 5. Cohort Analysis

In [ ]:
orders_c = orders_clean.copy()
orders_c["created_at"] = pd.to_datetime(orders_c["created_at"])
orders_c["order_month"] = orders_c["created_at"].dt.to_period("M")

# Cohort month = customer's first order month
orders_c["cohort_month"] = orders_c.groupby("customer_id")["order_month"].transform("min")

# Cohort index = months since acquisition
orders_c["cohort_index"] = (
    orders_c["order_month"].astype(int)
    - orders_c["cohort_month"].astype(int)
)

# Deduplicate to one active flag per customer per month
activity = orders_c.drop_duplicates(["customer_id", "order_month"])

cohort_counts = activity.groupby(["cohort_month", "cohort_index"])["customer_id"].nunique().unstack()
cohort_sizes  = cohort_counts.iloc[:, 0]
retention_matrix = cohort_counts.divide(cohort_sizes, axis=0).fillna(0).round(3)

print("Retention matrix shape:", retention_matrix.shape)
retention_matrix

## 6. Product Performance Fix (Q3 Output)

The `q3_product_performance.csv` query output had one null `product_id`.  
Fixed here and re-exported as `q3_products_clean.csv`.

In [ ]:
Q3_PATH      = r"data/query_outputs/q3_product_performance.csv"
Q3_OUT_PATH  = r"data/query_outputs/q3_products_clean.csv"

df_q3 = pd.read_csv(Q3_PATH)
print("Before fix:", df_q3.shape, "| null product_ids:", df_q3["product_id"].isna().sum())

df_q3["product_id"] = df_q3["product_id"].fillna(-1).astype(int)
df_q3["revenue"]    = pd.to_numeric(df_q3["revenue"], errors="coerce")
df_q3               = df_q3.sort_values("revenue", ascending=False).reset_index(drop=True)
df_q3["rank"]       = range(1, len(df_q3) + 1)

df_q3.to_csv(Q3_OUT_PATH, index=False)
print("After fix:", df_q3.shape)
df_q3